# Improving Retrieval with Query Transformations

<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/12-Improve_Query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## What This Notebook Covers
This notebook demonstrates advanced query transformation strategies that improve retrieval quality in RAG pipelines without changing the underlying vector store.

**Techniques covered:**
- **Multi-Step Query Decomposition** — breaks a complex query into sequential sub-questions, each answered in turn
- **SubQuestion Query Engine** — splits a query into per-source sub-questions answered in parallel
- **HyDE (Hypothetical Document Embeddings)** — generates a fake answer to improve embedding-based retrieval

## Learning Objectives
By the end of this notebook, you will be able to:
1. Configure and run a multi-step query engine with GPT and Gemini
2. Use a SubQuestion engine to query multiple document sources simultaneously
3. Apply HyDE to boost retrieval precision via hypothetical document generation
4. Inspect sub-question metadata and HyDE-generated hypothetical documents

## Prerequisites
- Familiarity with LlamaIndex (`VectorStoreIndex`, `QueryEngine`)
- OpenAI API key and Google API key configured
- Completed notebooks 03–11 (basic RAG, vector stores, prompt improvement)

## Install Packages and Setup Variables


In [ ]:
!pip install -qU llama-index==0.14.16 openai==2.30.0 google-genai==1.70.0 llama-index-embeddings-openai==0.5.2 \
                chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2 opentelemetry-api==1.38.0\
                llama-index-llms-openai==0.6.26 llama-index-finetuning==0.4.2 llama-index-embeddings-huggingface==0.7.0 \
                llama-index-llms-google-genai==0.9.1 llama-index-readers-web

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 62.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.4 MB/s eta 0:00

In [ ]:
import os

# Set the following API keys in the Python environment. They will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR-OPENAI-API-KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR-GOOGLE-API-KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", "")

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.
import nest_asyncio

nest_asyncio.apply()

# Initialize Models


In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load Indexes


In [ ]:
from huggingface_hub import hf_hub_download

vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

In [ ]:
!unzip -o vectorstore.zip

Archive:  vectorstore.zip
   creating: ai_tutor_knowledge/
   creating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/length.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/index_metadata.pickle  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/link_lists.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/header.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/data_level0.bin  
  inflating: ai_tutor_knowledge/chroma.sqlite3  


In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Create the vector index
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
vector_index = VectorStoreIndex.from_vector_store(vector_store)

# Multi-Step Query Engine


## GPT-4o-mini


In [ ]:
from llama_index.core.indices.query.query_transform.base import (
    StepDecomposeQueryTransform,
)

step_decompose_transform_gpt5 = StepDecomposeQueryTransform(verbose=True, llm=Settings.llm)

In [ ]:
from llama_index.core.query_engine.multistep_query_engine import MultiStepQueryEngine

# Default query engine
query_engine_gpt5_mini = vector_index.as_query_engine()

# Multi-step query engine
multi_step_query_engine = MultiStepQueryEngine(
    query_engine = query_engine_gpt5_mini,
    query_transform = step_decompose_transform_gpt5,
    index_summary = "Used to answer the Questions about RAG, Machine Learning, Deep Learning, and Generative AI, Note: Don't repeat the Same question",
)

# Query Dataset

## Default

In [ ]:
# Default query engine
query_engine = vector_index.as_query_engine()
res = query_engine.query("Write about Llama 3.1 Model, BERT and PEFT methods")
print(res.response)

I don’t have information here about a “Llama 3.1” model or about BERT — there are no details on those specific models in the material I have. I can, however, summarize what is available about PEFT-style methods for adapting LLaMA-family models and practical tooling around them.

PEFT methods and related tooling (summary)

- LoRA (via the 🤗 PEFT library)
  - The PEFT library provides support for fine-tuning LLaMA with the LoRA (Low-Rank Adaptation) approach. Example notebooks demonstrate LoRA-based fine-tuning with an easy-to-use UI and show how to load and run PEFT adapters (PeftModel).
  - Practical resources include notebooks for GPU-limited fine-tuning workflows (e.g., using xturing) and examples of integrating PEFT adapters with LangChain.

- Llama-Adapter (a prompt-based PEFT method)
  - Mechanism: the base LLaMA model is kept frozen and a small set of learnable adaptation prompts are prepended (prefixed) to input tokens at higher transformer layers. A zero-initialized attention m

In [ ]:
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 781b7b12-eca2-47c0-a66e-9d6be670e951
Title	 LLaMA
Text	 on how to fine-tune LLaMA model using LoRA method via the 🤗 PEFT library with intuitive UI. 🌎 - A [notebook](https://github.com/aws/amazon-sagemaker-examples/blob/main/introduction_to_amazon_algorithms/jumpstart-foundation-models/text-generation-open-llama.ipynb) on how to deploy Open-LLaMA model for text generation on Amazon SageMaker. 🌎 ## LlamaConfig[[autodoc]] LlamaConfig## LlamaTokenizer[[autodoc]] LlamaTokenizer    - build_inputs_with_special_tokens    - get_special_tokens_mask    - create_token_type_ids_from_sequences    - save_vocabulary## LlamaTokenizerFast[[autodoc]] LlamaTokenizerFast    - build_inputs_with_special_tokens    - get_special_tokens_mask    - create_token_type_ids_from_sequences    - update_post_processor    - save_vocabulary## LlamaModel[[autodoc]] LlamaModel    - forward## LlamaForCausalLM[[autodoc]] LlamaForCausalLM    - forward## LlamaForSequenceClassification[[autodoc]] LlamaForSequenceClassif

## GPT-4o-mini Multi-Step


In [ ]:
response = multi_step_query_engine.query("Write about Llama 3.1 Model, BERT and PEFT methods")
print(response.response)

> Current query: Write about Llama 3.1 Model, BERT and PEFT methods
> New query: How do Llama 3.1 and BERT fit into generative AI and RAG (retrieval-augmented generation) workflows, and what role do PEFT methods play in adapting these models?
> Current query: Write about Llama 3.1 Model, BERT and PEFT methods
> New query: How do LLaMA 3.1 and BERT participate in Retrieval-Augmented Generation (RAG) workflows?
> Current query: Write about Llama 3.1 Model, BERT and PEFT methods
> New query: How do LLaMA 3.1 and BERT fit into Retrieval‑Augmented Generation (RAG) workflows, and what does the provided material say about the role of PEFT methods in adapting these models?
Llama 3.1 model
- Classified in the material as an example of modern large generative language models (LLMs).
- Suited to generation and in‑context learning, and used as the generation/inference component in retrieval‑augmented pipelines: it conditions on retrieved passages (non‑parametric memory) to produce answers, helping

In [ ]:
for query, response in response.metadata['sub_qa']:
    print(f"**{query}**\n{response}\n")

**How do Llama 3.1 and BERT fit into generative AI and RAG (retrieval-augmented generation) workflows, and what role do PEFT methods play in adapting these models?**
Summary from the provided material:

- Where BERT fits
  - BERT is a discriminative language model that estimates conditional probabilities P(y|x) and is primarily suited to tasks like classification and sentence-level understanding.
  - Early RAG and GraphRAG work concentrated on improving pre‑training techniques for discriminative models such as BERT, i.e., using retrieval and graph structure to strengthen representations for discriminative tasks.

- Where LLaMA 3.1 (LLaMA family) fits
  - LLaMA is cited as an example of modern large language models (LLMs) that are generative and show strong in‑context learning and language understanding.
  - The research focus for RAG/GraphRAG has shifted toward enhancing information retrieval for LLMs like LLaMA, to support more complex tasks and reduce hallucinations by combining para

## Optional: Inspecting Multi-Step Query Decomposition

In [ ]:
# Show exactly how the multi-step engine broke the query into sub-questions
print("=== Multi-Step Query Decomposition ===\n")
try:
    sub_qa = response.metadata.get("sub_qa", [])
    print(f"Original query decomposed into {len(sub_qa)} sub-question(s):\n")
    for i, (sub_q, sub_a) in enumerate(sub_qa, 1):
        print(f"Step {i}: {sub_q}")
        print(f"  → {str(sub_a)[:200]}...")
        print()
    print("Each sub-question is answered independently, then combined into the final response.")
except Exception as e:
    print(f"Run the multi-step query cell first. ({e})")

=== Multi-Step Query Decomposition ===

Original query decomposed into 0 sub-question(s):

Each sub-question is answered independently, then combined into the final response.


In [ ]:
for src in response.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 2aa05360-f43a-4819-bce7-0acf7b897eab
Text	 Generative large language models are prone to producing outdated information or fabricating facts, although they were aligned with human preferences by reinforcement learning [1] or lightweight alternatives [2–5]. Retrieval-augmented generation (RAG) techniques address these issues by combining the strengths of pretraining and retrieval-based models, thereby providing a robust framework for enhancing model performance [6]. Furthermore, RAG enables rapid deployment of applications for specific organizations and domains without necessitating updates to the model parameters, as long as query-related documents are provided. Many RAG approaches have been proposed to enhance large language models (LLMs) through query-dependent retrievals [6–8]. A typical RAG workflow usually contains multiple intervening processing steps: query classification (determining whether retrieval is necessary for a given input query), retrieval (efficiently obtain

# Test gemini-2.5-flash Multi-Step


In [ ]:
from llama_index.core.indices.query.query_transform.base import StepDecomposeQueryTransform
from llama_index.core.query_engine.multistep_query_engine import MultiStepQueryEngine
from llama_index.llms.google_genai import GoogleGenAI

llm_gemini = GoogleGenAI(model="gemini-2.5-flash")

step_decompose_transform = StepDecomposeQueryTransform(llm=llm_gemini, verbose=True)

query_engine_gemini = vector_index.as_query_engine(llm=llm_gemini)

query_engine_gemini = MultiStepQueryEngine(
    query_engine=query_engine_gemini,
    query_transform=step_decompose_transform,
    index_summary="Used to answer the Questions about RAG, Machine Learning, Deep Learning, LLMs and Generative AI",
    num_steps=3,
)

In [ ]:
response_gemini = query_engine_gemini.query("Write about Llama 3.1 Model, BERT and PEFT")

> Current query: Write about Llama 3.1 Model, BERT and PEFT
> New query: What is Llama 3.1 Model?
> Current query: Write about Llama 3.1 Model, BERT and PEFT
> New query: What is BERT?
> Current query: Write about Llama 3.1 Model, BERT and PEFT
> New query: None


In [ ]:
print(response_gemini.response)

Llama 3.1 Model
- Llama 3.1 405B is a large-scale model developed by Meta. The 405B parameter variant was trained on over 15 trillion tokens using a training fleet of more than 16,000 H100 GPUs. It supports an extended 128K context length and exhibits enhanced abilities in reasoning and coding.
- Key capabilities:
  - RAG & tool use: supports zero-shot tool use and can develop agentic behaviors.
  - Multilingual processing: trained with roughly 50% multilingual tokens, enabling effective understanding and processing of multiple languages.
  - Programming and reasoning: strong at generating high-quality code and at logical reasoning, problem-solving, analysis, and inference.
- Evaluations: in benchmarks it outperformed certain contemporaries on mathematical and complex reasoning, multilingual tasks, and long-text processing (noted score: 95.2 points in zero-scrolls quality). Manual evaluations place its outputs comparable to top models like GPT-4 and Claude 3.5 Sonnet, with a slight gap

## Test Retriever on Multistep


In [ ]:
# import llama_index
# from llama_index.core.indices.query.schema import QueryBundle

# t = QueryBundle("How Retrieval Augmented Generation (RAG) work?")
# query_engine_gemini.retrieve(t)

## Subquestion Query Engine

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.question_gen.llm_generators import LLMQuestionGenerator
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})

question_gen = LLMQuestionGenerator.from_defaults(llm=llm)

query_engine = vector_index.as_query_engine()

query_engine_tools = [
    QueryEngineTool(
        query_engine=query_engine,
        metadata=ToolMetadata(
            name="LlamaIndex",
            description="Used to answer the Questions about RAG, Machine Learning, Deep Learning, and Generative AI. Note: Don't repeat the same question",
        ),
    ),
]

sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=query_engine_tools,
    question_gen=question_gen,
    use_async=True,
)

response = sub_question_engine.query("Write about Llama 3.1 Model, BERT and PEFT")

Generated 9 sub questions.
[LlamaIndex] Q: Give an overview of the Llama 3.1 model (purpose, scale variants, and typical use cases)
[LlamaIndex] Q: What are the architecture, training data, and training objectives for Llama 3.1?
[LlamaIndex] Q: What improvements or changes does Llama 3.1 introduce compared to earlier Llama versions?
[LlamaIndex] Q: Give an overview of BERT (purpose, architecture, and common pretraining tasks)
[LlamaIndex] Q: What are BERT's typical applications, strengths, and limitations?
[LlamaIndex] Q: Provide an overview of PEFT (parameter-efficient fine-tuning) and its main approaches (e.g., adapters, LoRA, prompt tuning)
[LlamaIndex] Q: Compare the different PEFT methods (mechanism, compute/memory tradeoffs, and when to prefer each)
[LlamaIndex] Q: How can PEFT methods be applied to Llama 3.1 and to BERT in practice (steps, libraries, and example use cases)?
[LlamaIndex] Q: Summarize practical recommendations: which model (Llama 3.1 vs BERT) and which PEFT method

In [ ]:
print(response.response)

Llama 3.1, BERT, and PEFT: concise guide

1) Llama 3.1 — overview, architecture, training, and use cases
- Purpose
  - A high-performance, open-source family of large autoregressive language models aimed at research, enterprise deployment, and customization. Designed to enable vendor independence, easier fine-tuning and on‑prem/cloud hosting, and competitive performance vs. closed models while lowering inference costs.

- Scale variants
  - 405B: Flagship scale (405 billion parameters). Trained at very large scale (over trillions of tokens). Offers a 128K token context window and top-tier capabilities in reasoning, coding, multilingual understanding, and long-text processing. Requires very large infrastructure for full-scale training, though targeted fine‑tuning workflows can use far fewer GPUs.
  - 70B: Middle-ground option; strong capabilities with far lower resource needs than 405B. Often used by organizations and cloud providers; distributed runs typically use high-speed interconne

# HyDE Transform


In [ ]:
query_engine = vector_index.as_query_engine()

In [ ]:
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine.transform_query_engine import TransformQueryEngine

hyde = HyDEQueryTransform(include_original=True)
hyde_query_engine = TransformQueryEngine(query_engine, hyde)

In [ ]:
response = hyde_query_engine.query("Write about Llama 3.1 Model, BERT and PEFT")

In [ ]:
print(response.response)

I can summarize what’s available about LLaMA and PEFT techniques, and note that there’s no information here about a “Llama 3.1” release or about BERT, so I can’t provide details for those beyond saying they aren’t covered in the available material.

LLaMA and PEFT (what’s available)
- LLaMA model family
  - Exposed configuration and model classes for various tasks (e.g., LlamaConfig, LlamaModel, LlamaForCausalLM, LlamaForSequenceClassification, LlamaForQuestionAnswering, LlamaForTokenClassification) and tokenizers (LlamaTokenizer, LlamaTokenizerFast) with standard helper methods for special tokens, token type ids, and saving vocabularies.
  - Flax implementations are available for causal LM use as well (FlaxLlamaModel, FlaxLlamaForCausalLM).

- PEFT methods for LLaMA
  - LoRA (Low-Rank Adaptation) is supported via the PEFT library and there are notebooks and examples showing how to fine-tune LLaMA with LoRA, including step-by-step guides and UIs.
  - Deployment examples exist (for inst

In [ ]:
for src in response.source_nodes:
    print("Node ID\t", src.node_id)
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

Node ID	 e1e3e842-7160-40c4-8e74-772fb8254f5e
Text	 # Llama-Adapter[Llama-Adapter](https://hf.co/papers/2303.16199) is a PEFT method specifically designed for turning Llama into an instruction-following model. The Llama model is frozen and only a set of adaptation prompts prefixed to the input instruction tokens are learned. Since randomly initialized modules inserted into the model can cause the model to lose some of its existing knowledge, Llama-Adapter uses zero-initialized attention with zero gating to progressively add the instructional prompts to the model.The abstract from the paper is:*We present LLaMA-Adapter, a lightweight adaption method to efficiently fine-tune LLaMA into an instruction-following model. Using 52K self-instruct demonstrations, LLaMA-Adapter only introduces 1.2M learnable parameters upon the frozen LLaMA 7B model, and costs less than one hour for fine-tuning on 8 A100 GPUs. Specifically, we adopt a set of learnable adaption prompts, and prepend them to the in

In [ ]:
query_bundle = hyde("Write about Llama 3.1 Model, BERT and PEFT")

In [ ]:
hyde_doc = query_bundle.embedding_strs[0]

In [ ]:
print("HyDE Document:\n", hyde_doc)

HyDE Document:
 Llama 3.1, BERT, and PEFT represent three complementary corners of modern NLP model design: a family of large autoregressive models for broad generative tasks, a foundational bidirectional encoder for representation and discriminative tasks, and a toolkit of efficient fine-tuning methods that make large models practical in applied settings. The following passage summarizes the essential architecture, training objectives, strengths, typical uses, and how these pieces fit together in production and research workflows.

Llama 3.1 (overview and engineering characteristics)
- Family and purpose: Llama 3.1 is an iteration in the LLaMA line of transformer-based large language models intended for high-quality text generation, instruction following, coding, summarization, and other open-ended tasks. It continues the design tradition of high-capacity autoregressive (decoder-only) transformers optimized for few-shot and instruction-tuned interactive use.
- Architecture and scale: 

## Optional: Comparing All Query Strategies

In [ ]:
# Compare default, multi-step, and HyDE on the same query
comparison_query = "How does LLaMA 2 improve over LLaMA 1?"
print(f"Query: {comparison_query}\n")
print("=" * 65)

strategies = [
    ("Default (direct)", lambda: vector_index.as_query_engine(similarity_top_k=3).query(comparison_query)),
    ("Multi-Step",       lambda: multi_step_query_engine.query(comparison_query)),
    ("HyDE",            lambda: hyde_query_engine.query(comparison_query)),
]

for name, fn in strategies:
    try:
        r = fn()
        ans = r.response if hasattr(r, 'response') else str(r)
        print(f"\n[{name}]")
        print(ans[:400] if ans else "(no response)")
    except Exception as e:
        print(f"\n[{name}] Error: {e}")

Query: How does LLaMA 2 improve over LLaMA 1?


[Default (direct)]
LLaMA 2 improves over the original LLaMA by introducing architectural tweaks—most notably Grouped Query Attention—and by being pre-trained on a much larger corpus (about 2 trillion tokens), resulting in a stronger pretrained model.
> Current query: How does LLaMA 2 improve over LLaMA 1?
> New query: What are the main improvements in LLaMA 2 over LLaMA 1 in terms of training data, fine-tuning, and safety?
> Current query: How does LLaMA 2 improve over LLaMA 1?
> New query: None

[Multi-Step]
I don’t have any information about LLaMA 1 or direct comparisons between LLaMA 1 and LLaMA 2, so I can’t reliably describe what improvements were made in training data, fine‑tuning, or safety.

If you can paste an excerpt that compares the two models or allow me to use external sources, I can summarize the differences.

[HyDE]
LLaMA 2 introduces architectural tweaks—most notably Grouped Query Attention—and is pre-trained on a much la